# Projeto 01 - Agente de Rotas para Comida Local (IA 2026-27)

*Inteligência Artificial - Licenciatura em Engenharia de Sistemas Informáticos (UPCA / EST)*

### Contexto
Uma plataforma digital liga **produtores** locais, **clientes** e **centros de recolha (armazéns)**.
Cada cliente faz uma encomenda e escolhe um **local de entrega**. Os produtos são recolhidos nos produtores,
passam obrigatoriamente pelo armazém (`producer > warehouse > delivery_location`) e são depois entregues,
respeitando:
* a **janela de disponibilidade** do produto no produtor (`availability_start / availability_end`);
* a **janela de entrega** no local de destino (`delivery_start / delivery_end`);
* a **capacidade** dos veículos.

O objetivo do projeto é desenvolver um agente que escolha as melhores rotas de recolha e entrega usando
**pesquisa informada** (como Greedy, A*), **pesquisa local** (como Hill-Climbing) e **CSP** (biblioteca `python-constraint`).

### Equipa (Grupo G04)
| Nome | Número |
|------|--------|
| Diogo Apolinário | 6731 |
| Raúl Ribeiro | 22552 |
| Miguel Correia | 31028 |
| José Ferreira | 31556 |

**Repositório GitHub:** https://github.com/JoseFerreira31556/IA26_P01_G04.git

### Este notebook - Entrega 1: Leitura dos ficheiros e contagem de linhas
O pandas lê os ficheiros CSV do dataset, conta as linhas de cada um e imprime esse valor.

In [553]:
# Importar bibliotecas
import pandas
import os

In [554]:
path = "./data/data_tiny_20261020/"  # path to the folder containing the CSV files
files_list = sorted(os.listdir(path)) # list of CSV files in the folder

def read_csv_files(path, files_list):    
    data = {} #dicionário com a informação dos ficheiros

    for file in files_list:
        if not file.endswith(".csv"): continue

        # Read the CSV file into a pandas dataframe
        rf = pandas.read_csv(os.path.join(path,file))

        name = file.replace(".csv", "")   
        data[name] = rf
        
        #rows -> line number
        #columns -> column number

        # rows, columns = rf.shape

        # print(file + ": " + str(rows) + " rows")  

    return data

data = read_csv_files(path, files_list)

### Verificação de incongruências nos dados
Pretende-se com este bloco verificar se existem valores em falta, linhas duplicadas e se o tipo corresponde à informação observada na coluna. 
Foram usados os seguintes métodos do pandas:
*  **items()**: permite tirar do dicionário a key e o conteúdo;
*  **isna()**: verifica se existe alguma célula vazia, atribuindo um True em caso verdadeiro e um False caso não esteja
*  **duplicated()**: verificação da existência de duplicados
*  **sum()**: soma dos valores que deu na tabela. Considera o True como 1 e o False como 0
*  **duplicated(subset=[])**: permite escolher as colunas que se querem comparar

In [555]:
#data.items() gives the name of the file and it's inside information
for name, table in data.items():         
    #Variable to check for empty cells
    sumEmptyCells = table.isna().sum().sum()

    #Variable to check for duplicated cells
    duplicates = table.duplicated().sum()   

    #Verifying if there is empty cells
    if(sumEmptyCells != 0): print("Missing a value in file " + name)

    #Verifying if there is duplicated cells
    if(duplicates != 0): print("File " + name + " with duplicated information") 

    #Variable that checks only the first column of the table
    tableIdColumn = table.columns[0]  

    isId = tableIdColumn.endswith("_id")
    if(isId == False): continue

    if(name == "distance_matrix"):
        duplicatesDM = table.duplicated(subset=["origin_location_id", "destination_location_id"]).sum()
        if(duplicatesDM != 0 ) : 
            print("Duplicated lines in file " + name)            
        continue

    #Variable that counts the number of duplicated id's
    duplicatedIds= table[tableIdColumn].duplicated().sum()

    #Verifying if there is duplicated id'S
    if(duplicatedIds != 0): print("File " + name + " with duplicated id")   
                             


### Conversão de datas em string para datetime
Pretende-se com este bloco converter as datas em datetime. Para isso foi usado um método do pandas: 
*  **pandas.to_datetime()**: converte as variáveis para o tipo *datetime(us)*

In [556]:
for name, table in data.items(): 
    #Converting dates (strings to datetime)
    for column in table.columns:
        if(column.endswith(("_end", "_start", "_datetime"))):
            dateColumns = table[column]
            table[column] = pandas.to_datetime(dateColumns)          
    

### Verificação da integridade dos dados

Pretende-se verificar se os identificadores referenciados entre os ficheiros CSV existem nas respetivas tabelas e se a matriz de distâncias contém todas as ligações esperadas.

In [557]:
# Verificar os IDs referenciados entre tabelas
relations = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "delivery_location_id", "delivery_locations", "delivery_location_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("products", "producer_id", "producers", "producer_id"),
    ("producers", "locality_id", "localities", "locality_id"),
    ("vehicles", "center_id", "collection_centers", "center_id"),
    ("distance_matrix", "origin_location_id", "locations", "location_id"),
    ("distance_matrix", "destination_location_id", "locations", "location_id")
]

for source, column, target, targetColumn in relations:
    invalidIds = set(data[source][column]) - set(data[target][targetColumn])

    if(len(invalidIds) != 0):
        print("Invalid IDs in " + source + ": " + str(invalidIds))


# Verificar as referências da tabela locations
references = {
    "producer": ("producers", "producer_id"),
    "collection_center": ("collection_centers", "center_id"),
    "delivery_location": ("delivery_locations", "delivery_location_id")
}

for locationType, (tableName, idColumn) in references.items():
    locations = data["locations"]
    filteredLocations = locations[locations["location_type"] == locationType]

    invalidIds = set(filteredLocations["ref_id"]) - set(data[tableName][idColumn])

    if(len(invalidIds) != 0):
        print("Invalid references in locations: " + str(invalidIds))


# Verificar a matriz de distâncias
locationsIds = set(data["locations"]["location_id"])
expectedPairs = set()

for origin in locationsIds:
    for destination in locationsIds:
        if(origin != destination):
            expectedPairs.add((origin, destination))

distanceMatrix = data["distance_matrix"]

actualPairs = list(zip(distanceMatrix["origin_location_id"], distanceMatrix["destination_location_id"]))

print("Expected pairs:", len(expectedPairs))
print("Existing pairs:", len(actualPairs))
print("Duplicated pairs:", len(actualPairs) - len(set(actualPairs)))
print("Missing pairs:", len(expectedPairs - set(actualPairs)))
print("Invalid pairs:", len(set(actualPairs) - expectedPairs))

Expected pairs: 72
Existing pairs: 72
Duplicated pairs: 0
Missing pairs: 0
Invalid pairs: 0


### Verificação das janelas temporais

Pretende-se verificar a coerência dos horários de disponibilidade dos produtos e das janelas de entrega, bem como identificar as restrições temporais definidas nas regras do projeto.

In [558]:
# Verificar janelas temporais
orderItems = data["order_items"]

invalidAvailability = orderItems[
    orderItems["availability_start"] >= orderItems["availability_end"]
]

invalidDelivery = orderItems[
    orderItems["delivery_start"] >= orderItems["delivery_end"]
]

print("Invalid availability windows:", len(invalidAvailability))
print("Invalid delivery windows:", len(invalidDelivery))


# Verificar a sequência temporal
invalidSequence = orderItems[
    orderItems["availability_start"] > orderItems["delivery_end"]
]

print("Invalid time sequences:", len(invalidSequence))


# Consultar as regras temporais
rules = data["rules"]

temporalRules = ["flow", "window_semantics", "max_warehouse_dwell_h"]

for rule in temporalRules:
    result = rules[rules["rule"] == rule]
    print(rule + ":", result["value"].iloc[0])

Invalid availability windows: 0
Invalid delivery windows: 0
Invalid time sequences: 0
flow: producer>warehouse>delivery_location
window_semantics: service_start
max_warehouse_dwell_h: 24


### Pesos vs Capacidades

Dentro da tabela "products" foram escolhidas as colunas "product_id" e "unit_weight_kg" para fazer merge à tabela "orderItems". Escolheram-se estas duas colunas pelo facto de "product_id" ser comum nas duas tabelas e "unit_weight_kg" para permitir realizar a verificação dos valores na coluna "weight_kg" da tabela "orderItems". Esta verificação é feita através de uma multiplicação entre os valores da coluna "quantity" e "unit_weight_kg". 

In [ ]:
productsTableModified = data["products"][["product_id", "unit_weight_kg"]]

#Merged the two table by column "product_id"
tableJunctionP_OI = pandas.merge(orderItems, productsTableModified, on="product_id")

#New column with the values of the multiplication
tableJunctionP_OI["weight_verification"] = tableJunctionP_OI["quantity"] * tableJunctionP_OI["unit_weight_kg"] 

#New column with True or False. True for different values and False for equal values
tableJunctionP_OI["weight_mismatch"] = tableJunctionP_OI["weight_verification"] != tableJunctionP_OI["weight_kg"]

valuesSum = tableJunctionP_OI["weight_mismatch"].sum()

if(valuesSum == 0):
    print("All the weights are well calculated")
else:
    print("There are " + str(valuesSum) + " lines that have wrong weights")
    #Prints the lines that mismatch
    print(tableJunctionP_OI[tableJunctionP_OI["weight_mismatch"]])

#Total weight calculation
totalWeight = tableJunctionP_OI["weight_kg"].sum()

print("Total Weight: " + str(totalWeight) + " kg")

#Total fleet capacity
totalFleetCap = data["vehicles"]["capacity_kg"].sum()

print("Total Fleet Capacity: " + str(totalFleetCap) + " kg")

#Total warehouse capacity
totalWarehouseCap = data["collection_centers"]["capacity_kg"].sum()

print("Total Warehouse Capacity: " + str(totalWarehouseCap) + " kg")

#Weight by order
groupByOrderID = data["order_items"].groupby("order_id")["weight_kg"].sum()

#Does the heavier order fits the heavier vehicle?
#Heavier Order
print("Biggest Order: " + str(groupByOrderID.max().round()) + " kg")  

#Vehicle with more capacity
print("Bigger Vehicle Capacity: ")
print(data["vehicles"][["vehicle_id", "capacity_kg"]].max())

#Weight by day



All the weights are well calculated
Total Weight: 270.0 kg
Total Fleet Capacity: 350.0 kg
Total Warehouse Capacity: 5000.0 kg
Biggest Order: 51.0 kg
Bigger Vehicle Capacity: 
vehicle_id       2.0
capacity_kg    200.0
dtype: float64
